# Task 4: 补上 ΔΔG 与结构失稳特征

**目标**: 用 ESM2-650M 零样本方法计算突变 ΔΔG（折叠自由能变化），填补 `features_v3.csv` 中全是 NaN 的 `ddg` 列。

**方法**: Masked Marginal（伪对数似然比）
- 对每个变体，在 WT 序列的突变位置放 `<mask>`
- ESM2 输出该位置的氨基酸概率分布
- ΔΔG_proxy = log P(WT_AA | context) − log P(MT_AA | context)
- **正值** → 突变不利（去稳定化）；**负值** → 突变有利（稳定化）

**注意**: 需要使用 GPU，ESM2-650M 半精度约需 1.3GB 显存。

In [2]:
import numpy as np, pandas as pd, re, time, os, warnings
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import torch
from transformers import EsmForMaskedLM, EsmTokenizer
warnings.filterwarnings("ignore")

BASE_PATH = "/mnt/volume6/czj/labLGN/LabLZ/xgboost_trial/"
MAX_SEQ_LEN = 1022
BATCH_SIZE = 8

# ===== 加载数据 =====
df_main = pd.read_csv("/mnt/volume6/czj/labLGN/LabLZ/data_preparation/cell2024_model_single_subst.csv")
print(f"数据加载完毕: {len(df_main)} 行")

# ===== 加载 ESM2 模型 =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

tokenizer = EsmTokenizer.from_pretrained("/mnt/volume6/czj/labLGN/LabLZ/models/esm2_650M")
model = EsmForMaskedLM.from_pretrained("/mnt/volume6/czj/labLGN/LabLZ/models/esm2_650M").eval().to(device)
if device.type == "cuda":
    model = model.half()
    print(f"模型已加载 (半精度), GPU 显存: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")
else:
    print("模型已加载 (CPU — 会比较慢)")

MASK_ID = tokenizer.mask_token_id


数据加载完毕: 2179 行
Device: cuda


Loading weights:   0%|          | 0/539 [00:00<?, ?it/s]

模型已加载 (半精度), GPU 显存: 2.7 GB


## 4a. 准备 masked 序列

对每个变体，解析突变位点，在序列对应位置替换为 `<mask>`，同时截断超长序列到突变位点附近。

In [4]:
# ===== 解析突变 =====
def parse_mutation(mut_str):
    if not isinstance(mut_str, str): return None, None, None
    m = re.match(r'^([A-Z])(\d+)([A-Z])$', mut_str.strip())
    return (m.group(1), int(m.group(2)), m.group(3)) if m else (None, None, None)

def truncate_around_pos(seq, pos):
    if len(seq) <= MAX_SEQ_LEN:
        return seq, pos - 1
    half = MAX_SEQ_LEN // 2
    end = min(len(seq), pos + half)
    start = max(0, end - MAX_SEQ_LEN)
    return seq[start:start + MAX_SEQ_LEN], pos - start - 1

df_ddg = df_main[["Gene", "Variant", "Mutation_used", "sequence"]].copy()
df_ddg["wt_aa"], df_ddg["pos"], df_ddg["mt_aa"] = zip(
    *df_ddg["Mutation_used"].apply(parse_mutation))

valid = df_ddg["wt_aa"].notna() & df_ddg["sequence"].notna()
print(f"有效突变: {valid.sum()} / {len(df_ddg)}")
df_ddg = df_ddg[valid].copy()

# 构建 masked 序列
masked_seqs, valid_indices = [], []
skip = {"wt_mismatch": 0, "pos_oob": 0, "ok": 0}

for idx, row in df_ddg.iterrows():
    seq, wt_aa, pos = row["sequence"], row["wt_aa"], row["pos"]
    if pos < 1 or pos > len(seq): skip["pos_oob"] += 1; continue
    if seq[pos-1] != wt_aa: skip["wt_mismatch"] += 1; continue
    trunc_seq, trunc_pos = truncate_around_pos(seq, pos)
    if trunc_pos < 0 or trunc_pos >= len(trunc_seq): skip["pos_oob"] += 1; continue
    masked = trunc_seq[:trunc_pos] + "<mask>" + trunc_seq[trunc_pos+1:]
    masked_seqs.append(masked)
    valid_indices.append(idx)
    skip["ok"] += 1

print(f"就绪序列: {len(masked_seqs)} (跳过: {skip})")
df_ddg = df_ddg.loc[valid_indices].copy()
# KEY 使用 Gene + Mutation_used（而非 Variant），因为 Mutation_used 是实际用于建模的突变
KEY_ORDER = (df_ddg["Gene"] + "||" + df_ddg["Mutation_used"]).tolist()

有效突变: 2179 / 2179
就绪序列: 2179 (跳过: {'wt_mismatch': 0, 'pos_oob': 0, 'ok': 2179})


## 4b. 批量推理计算 ΔΔG

In [6]:
# ===== ΔΔG 计算 =====
AA_TO_TOKEN = {aa: tokenizer.convert_tokens_to_ids(aa) for aa in "ACDEFGHIKLMNPQRSTVWY"}
print(AA_TO_TOKEN)  # 应看到 A=5, L=4, C=23 

ddg_scores = np.full(len(masked_seqs), np.nan, dtype=np.float32)

@torch.inference_mode()
def compute_ddg_batch(batch_seqs, batch_wt, batch_mt):
    enc = tokenizer(batch_seqs, return_tensors="pt", padding=True,
                    truncation=True, max_length=MAX_SEQ_LEN+4)
    ids = enc["input_ids"].to(device); attn = enc["attention_mask"].to(device)
    if device.type == "cuda":
        with torch.autocast("cuda"):
            logits = model(input_ids=ids, attention_mask=attn).logits.float()
    else:
        logits = model(input_ids=ids, attention_mask=attn).logits.float()
    mask_pos = (ids == MASK_ID).nonzero(as_tuple=False)
    results = np.full(len(batch_seqs), np.nan, dtype=np.float32)
    pos_dict = {}
    for r, c in mask_pos:
        if r.item() not in pos_dict: pos_dict[r.item()] = c.item()
    for i in range(len(batch_seqs)):
        if i not in pos_dict: continue
        p = pos_dict[i]
        wt, mt = batch_wt[i], batch_mt[i]
        if wt in AA_TO_TOKEN and mt in AA_TO_TOKEN:
            results[i] = logits[i,p,AA_TO_TOKEN[wt]].item() - logits[i,p,AA_TO_TOKEN[mt]].item()
    return results

print(f"开始计算 {len(masked_seqs)} 个变体的 ΔΔG (batch_size={BATCH_SIZE}) ...")
t0 = time.time()
n_batches = (len(masked_seqs) + BATCH_SIZE - 1) // BATCH_SIZE
for i in range(0, len(masked_seqs), BATCH_SIZE):
    batch_seqs = masked_seqs[i:i+BATCH_SIZE]
    batch_wt = df_ddg["wt_aa"].iloc[i:i+BATCH_SIZE].tolist()
    batch_mt = df_ddg["mt_aa"].iloc[i:i+BATCH_SIZE].tolist()
    ddg_scores[i:i+BATCH_SIZE] = compute_ddg_batch(batch_seqs, batch_wt, batch_mt)
    bn = i // BATCH_SIZE + 1
    if bn % 50 == 0 or bn == 1 or bn == n_batches:
        elapsed = time.time() - t0
        print(f"  Batch {bn}/{n_batches} ({bn/n_batches*100:.0f}%) "
              f"耗时={elapsed:.0f}s 预计剩余={elapsed/bn*(n_batches-bn):.0f}s")

elapsed = time.time() - t0
n_valid = np.isfinite(ddg_scores).sum()
print(f"\nΔΔG 计算完成! 总耗时 {elapsed:.0f}s ({elapsed/len(masked_seqs):.2f}s/变体)")
print(f"覆盖率: {n_valid}/{len(ddg_scores)} ({n_valid/len(ddg_scores)*100:.1f}%)")

ddg_finite = ddg_scores[np.isfinite(ddg_scores)]
print(f"ΔΔG 统计: mean={ddg_finite.mean():.3f}, std={ddg_finite.std():.3f}, "
      f"min={ddg_finite.min():.3f}, max={ddg_finite.max():.3f}")

# 保存
df_out = pd.DataFrame({"KEY": KEY_ORDER, "ddg_esm2": ddg_scores})
df_out.to_csv(BASE_PATH + "ddg_esm2.csv", index=False)
print("已保存 ddg_esm2.csv")


{'A': 5, 'C': 23, 'D': 13, 'E': 9, 'F': 18, 'G': 6, 'H': 21, 'I': 12, 'K': 15, 'L': 4, 'M': 20, 'N': 17, 'P': 14, 'Q': 16, 'R': 10, 'S': 8, 'T': 11, 'V': 7, 'W': 22, 'Y': 19}
开始计算 2179 个变体的 ΔΔG (batch_size=8) ...
  Batch 1/273 (0%) 耗时=7s 预计剩余=1898s
  Batch 50/273 (18%) 耗时=13s 预计剩余=57s
  Batch 100/273 (37%) 耗时=18s 预计剩余=32s
  Batch 150/273 (55%) 耗时=24s 预计剩余=20s
  Batch 200/273 (73%) 耗时=30s 预计剩余=11s
  Batch 250/273 (92%) 耗时=36s 预计剩余=3s
  Batch 273/273 (100%) 耗时=39s 预计剩余=0s

ΔΔG 计算完成! 总耗时 39s (0.02s/变体)
覆盖率: 2179/2179 (100.0%)
ΔΔG 统计: mean=5.285, std=4.095, min=-5.716, max=14.969
已保存 ddg_esm2.csv


## 4c. 将 ΔΔG 加入特征集，重新二分类评估

In [7]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

# ===== 将 ΔΔG 映射到全量 2179 行 =====
ddg_map = dict(zip(KEY_ORDER, ddg_scores))
# KEY 使用 Gene + Mutation_used，与上方 KEY_ORDER 一致
full_keys = (df_main["Gene"] + "||" + df_main["Mutation_used"]).tolist()
ddg_full = np.array([ddg_map.get(k, np.nan) for k in full_keys], dtype=np.float32)

# ===== 特征矩阵 =====
df_feat = pd.read_csv(BASE_PATH + "features_v3.csv")
ID_COLS = ["KEY", "Gene", "reloc_v3"]
NAN_PLACEHOLDERS = ["ddg", "plddt_diff"]
exclude_cols = ID_COLS + NAN_PLACEHOLDERS
feat_cols = [c for c in df_feat.columns if c not in exclude_cols]

X_base = df_feat[feat_cols].values.astype(np.float32)
X_with_ddg = np.hstack([X_base, ddg_full.reshape(-1, 1)])

y_bin = (df_feat["reloc_v3"].values > 0).astype(int)
groups = df_feat["Gene"].values

print(f"特征矩阵: {X_with_ddg.shape}")
print(f"ddg 列 NaN 数: {np.isnan(ddg_full).sum()}/{len(ddg_full)}")

# ===== CV 对比: 有/无 ΔΔG =====
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
oof_ddg = np.zeros(len(y_bin), dtype=np.float32)
oof_noddg = np.zeros(len(y_bin), dtype=np.float32)

for fold, (tr_idx, te_idx) in enumerate(cv.split(X_with_ddg, y_bin, groups)):
    y_tr = y_bin[tr_idx]

    # --- 有 ΔΔG ---
    imp = SimpleImputer(strategy="median"); scl = StandardScaler()
    X_tr = scl.fit_transform(imp.fit_transform(X_with_ddg[tr_idx])).astype(np.float32)
    X_te = scl.transform(imp.transform(X_with_ddg[te_idx])).astype(np.float32)
    sw = compute_sample_weight("balanced", y_tr)
    m = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                      subsample=0.8, colsample_bytree=0.5,
                      objective="binary:logistic", eval_metric="aucpr",
                      random_state=42, n_jobs=-1, tree_method="hist")
    m.fit(X_tr, y_tr, sample_weight=sw, verbose=False)
    oof_ddg[te_idx] = m.predict_proba(X_te)[:, 1]

    # --- 无 ΔΔG ---
    imp_b = SimpleImputer(strategy="median"); scl_b = StandardScaler()
    X_tr_b = scl_b.fit_transform(imp_b.fit_transform(X_base[tr_idx])).astype(np.float32)
    X_te_b = scl_b.transform(imp_b.transform(X_base[te_idx])).astype(np.float32)
    sw_b = compute_sample_weight("balanced", y_tr)
    m_b = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.5,
                        objective="binary:logistic", eval_metric="aucpr",
                        random_state=42, n_jobs=-1, tree_method="hist")
    m_b.fit(X_tr_b, y_tr, sample_weight=sw_b, verbose=False)
    oof_noddg[te_idx] = m_b.predict_proba(X_te_b)[:, 1]

    y_te = y_bin[te_idx]
    print(f"  Fold {fold}: +ddg AUROC={roc_auc_score(y_te, oof_ddg[te_idx]):.4f}  "
          f"baseline AUROC={roc_auc_score(y_te, oof_noddg[te_idx]):.4f}")

# ===== 汇总 =====
auroc_ddg = roc_auc_score(y_bin, oof_ddg)
auprc_ddg = average_precision_score(y_bin, oof_ddg)
auroc_no = roc_auc_score(y_bin, oof_noddg)
auprc_no = average_precision_score(y_bin, oof_noddg)
br = float(y_bin.sum() / len(y_bin))

print(f"\n{'='*60}")
print(f"  TASK 4 最终结果: v3 + ΔΔG vs v3 基线 (同折)")
print(f"  n={len(y_bin)}, 正例={int(y_bin.sum())}, base_rate={br:.4f}")
print(f"  {'指标':<15s} {'v3 基线':>12s} {'v3 + ΔΔG':>12s} {'增量':>12s}")
print(f"  {'AUROC':<15s} {auroc_no:>12.4f} {auroc_ddg:>12.4f} {auroc_ddg-auroc_no:>+12.4f}")
print(f"  {'AUPRC':<15s} {auprc_no:>12.4f} {auprc_ddg:>12.4f} {auprc_ddg-auprc_no:>+12.4f}")
print(f"{'='*60}")


特征矩阵: (2179, 1289)
ddg 列 NaN 数: 0/2179
  Fold 0: +ddg AUROC=0.6153  baseline AUROC=0.6066
  Fold 1: +ddg AUROC=0.6122  baseline AUROC=0.6373
  Fold 2: +ddg AUROC=0.5645  baseline AUROC=0.5552
  Fold 3: +ddg AUROC=0.6190  baseline AUROC=0.6064
  Fold 4: +ddg AUROC=0.5889  baseline AUROC=0.5685

  TASK 4 最终结果: v3 + ΔΔG vs v3 基线 (同折)
  n=2179, 正例=235, base_rate=0.1078
  指标                     v3 基线     v3 + ΔΔG           增量
  AUROC                 0.5940       0.5983      +0.0043
  AUPRC                 0.1604       0.1621      +0.0017


## 4d. 特征重要性（含 ΔΔG 的排名）

In [8]:
print("\n--- 特征重要性 (v3 + ΔΔG, fit on all data 仅用于排名) ---")
imp_all = SimpleImputer(strategy="median"); scl_all = StandardScaler()
X_all_ddg = scl_all.fit_transform(imp_all.fit_transform(X_with_ddg)).astype(np.float32)
sw_ratio = (y_bin==0).sum() / max((y_bin==1).sum(), 1)

xgb_fi = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                       subsample=0.8, colsample_bytree=0.5,
                       scale_pos_weight=sw_ratio,
                       objective="binary:logistic", eval_metric="aucpr",
                       random_state=42, n_jobs=-1, tree_method="hist")
xgb_fi.fit(X_all_ddg, y_bin, verbose=False)

all_names = feat_cols + ["ddg_esm2"]
imps = xgb_fi.feature_importances_
idx_sorted = np.argsort(imps)[::-1]

print("\nTop-30 特征重要性:")
for rank, i in enumerate(idx_sorted[:30]):
    val = imps[i]; bar = "█" * int(val * 2000)
    print(f"  {rank+1:2d}. {all_names[i]:30s}  {val:.5f}  {bar}")

ddg_idx = len(feat_cols)
ddg_rank = np.where(idx_sorted == ddg_idx)[0][0] + 1
ddg_imp = imps[ddg_idx]
print(f"\n  ΔΔG (ddg_esm2): 排名={ddg_rank}/{len(all_names)}, "
      f"重要性={ddg_imp:.5f}")
print(f"  → {'ΔΔG 进入了 top-30!' if ddg_rank <= 30 else 'ΔΔG 排名较后，增量有限'}")



--- 特征重要性 (v3 + ΔΔG, fit on all data 仅用于排名) ---

Top-30 特征重要性:
   1. esm_898                         0.00488  █████████
   2. esm_533                         0.00468  █████████
   3. esm_1168                        0.00453  █████████
   4. esm_413                         0.00419  ████████
   5. ddg_esm2                        0.00402  ████████
   6. esm_583                         0.00384  ███████
   7. esm_1099                        0.00361  ███████
   8. esm_1142                        0.00355  ███████
   9. esm_1160                        0.00351  ███████
  10. esm_566                         0.00350  ██████
  11. esm_539                         0.00349  ██████
  12. esm_724                         0.00348  ██████
  13. esm_1194                        0.00332  ██████
  14. esm_1178                        0.00332  ██████
  15. esm_356                         0.00327  ██████
  16. esm_572                         0.00311  ██████
  17. esm_1109                        0.00309  ██████
 

## 4e. ΔΔG 与 AlphaMissense 的相关性分析

检查 ESM2 零样本 ΔΔG 与 AlphaMissense 分数之间的相关性——两者是否捕捉了相似或互补的信号？

In [12]:
from scipy.stats import pearsonr, spearmanr

# ===== 计算 ddg_esm2 与 AlphaMissense 的相关性 =====
am_scores = df_main["AlphaMissense score"].values.astype(np.float64)

# 两者都有效的行
mask_corr = np.isfinite(ddg_full) & np.isfinite(am_scores)
n_corr = mask_corr.sum()
print(f"有效样本 (两者均非NaN): {n_corr}/{len(ddg_full)}")

# Pearson 相关系数
r_pearson, p_pearson = pearsonr(ddg_full[mask_corr], am_scores[mask_corr])
# Spearman 秩相关系数
r_spearman, p_spearman = spearmanr(ddg_full[mask_corr], am_scores[mask_corr])

print(f"\nddg_esm2 vs AlphaMissense score 相关性:")
print(f"  Pearson  r = {r_pearson:+.4f}  (p = {p_pearson:.2e})")
print(f"  Spearman r = {r_spearman:+.4f}  (p = {p_spearman:.2e})")

# 按标签分别看
print(f"\n按标签分组:")
for label, name in [(0, "负例 (不重定位)"), (1, "正例 (重定位)")]:
    mask_label = mask_corr & (y_bin == label)
    n_label = mask_label.sum()
    if n_label > 5:
        rp, pp = pearsonr(ddg_full[mask_label], am_scores[mask_label])
        rs, ps = spearmanr(ddg_full[mask_label], am_scores[mask_label])
        print(f"  {name:20s} (n={n_label:4d}): Pearson r={rp:+.4f}, Spearman r={rs:+.4f}")

# 解读
print(f"\n{'='*60}")
abs_r = abs(r_spearman)
if abs_r < 0.3:
    print(f"  |r| = {abs_r:.3f} < 0.3 → ΔΔG 与 AlphaMissense 几乎正交")
    print(f"  → 两者捕捉的几乎是完全不同的信号，组合使用价值高")
elif abs_r < 0.6:
    print(f"  0.3 ≤ |r| = {abs_r:.3f} < 0.6 → 中等相关")
    print(f"  → 部分重叠，但仍有互补空间")
else:
    print(f"  |r| = {abs_r:.3f} ≥ 0.6 → 较强相关")
    print(f"  → ΔΔG 与 AlphaMissense 可能捕捉了相似的信号")
print(f"{'='*60}")


有效样本 (两者均非NaN): 2053/2179

ddg_esm2 vs AlphaMissense score 相关性:
  Pearson  r = +0.7108  (p = 1.15e-315)
  Spearman r = +0.7173  (p = 0.00e+00)

按标签分组:
  负例 (不重定位)            (n=1832): Pearson r=+0.7095, Spearman r=+0.7213
  正例 (重定位)             (n= 221): Pearson r=+0.6535, Spearman r=+0.5956

  |r| = 0.717 ≥ 0.6 → 较强相关
  → ΔΔG 与 AlphaMissense 可能捕捉了相似的信号
